# Concept 2: Why Brute Force Doesn't Scale

Times a real similarity search loop over 20 snippets — the same cost that repeats, unchanged, on every query.

## Setup
Store your key in Colab **Secrets** (key icon, left sidebar) as `OPENAI_API_KEY`.

In [ ]:
!pip install -q openai numpy

In [ ]:
from openai import OpenAI
from google.colab import userdata
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

## Snippets
Colab starts with an empty filesystem each session, so we fetch the 20 sample snippets from GitHub first.

In [ ]:
import urllib.request
url = "https://raw.githubusercontent.com/Roopesht/codeexamples/main/genai/llm/705/docs/snippets.txt"
snippets = urllib.request.urlopen(url).read().decode().strip().split("\n")

In [ ]:
import numpy as np
def embed(text):
    return client.embeddings.create(model="text-embedding-3-small", input=text).data[0].embedding
def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
import time
query_vector = embed("What are baby dogs called?")
start = time.time()
scores = [cosine(query_vector, embed(s)) for s in snippets]
print(f"{time.time() - start:.2f}s for {len(snippets)} items")

## How to Extend
Duplicate `snippets` a few times (e.g. `snippets * 10`) and re-run — watch the time roughly scale with the count. Now imagine that same loop over a million real documents, on every single search.